In [1]:
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from rapidfuzz import process

d:\PROJECT\BookSage AI\book_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = SentenceTransformer("models\\custom_embedding_model")
index = faiss.read_index("models\\book_index.faiss")

W0309 19:00:55.393000 18952 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [3]:
with open("models\\book_metadata.pkl", "rb") as f:
    df = pickle.load(f)

In [4]:
df.reset_index(drop=True, inplace=True)

In [5]:
titles = df["Title"].tolist()

In [6]:
def find_closest_title(user_input):
    match = process.extractOne(user_input, titles)
    
    if match is None:
        return None
    
    return match[0]

In [7]:
def recommend_books(book_title, top_k=5):

    if top_k > 5:
        top_k = 5

    matched_title = find_closest_title(book_title)

    if matched_title is None:
        print("No matching book found.")
        return

    print(f"\nMatched Book: {matched_title}")

    book_row = df[df["Title"] == matched_title]

    if book_row.empty:
        print("Book not found in dataset.")
        return

    book_row = book_row.iloc[0]
    book_idx = df[df["Title"] == matched_title].index[0]

    query_text = book_row["document"]
    query_embedding = model.encode([query_text], convert_to_numpy=True)

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, top_k + 1)

    print("\nSelected Book Details:")
    print("-----------------------------")
    print("Title:", book_row["Title"])
    print("Author:", book_row["Author"])
    print("Genre:", book_row["Main Genre"])
    print("Rating:", book_row["Rating"])
    print("Price:", book_row["Price"])

    print("\n Recommended Books:")
    print("=============================")

    count = 0
    for idx in indices[0]:

        if idx == book_idx:
            continue

        book = df.iloc[idx]

        print("\n-----------------------------")
        print("Title:", book["Title"])
        print("Author:", book["Author"])
        print("Genre:", book["Main Genre"])
        print("Rating:", book["Rating"])
        print("Price:", book["Price"])

        count += 1
        if count >= top_k:
            break

In [8]:
if __name__ == "__main__":
    user_book = input("Enter book title: ")
    
    try:
        k = int(input("How many recommendations (max 5)? "))
    except ValueError:
        print("Invalid number. Defaulting to 5.")
        k = 5

    recommend_books(user_book, k)


Matched Book: Built to Move: The 10 essential habits that will help you live a longer, healthier life

Selected Book Details:
-----------------------------
Title: Built to Move: The 10 essential habits that will help you live a longer, healthier life
Author: Juliet Starrett
Genre: Sports
Rating: 4.6
Price: 589.41

 Recommended Books:

-----------------------------
Title: Built to Move: The 10 Essential Habits to Help you Move Freely and Live Fully
Author: Juliet Starrett
Genre: Sports
Rating: 4.6
Price: 835.0

-----------------------------
Title: Swim Like A Pro: How to Swim Faster & Smarter With A Holistic Training Guide
Author: Fares Ksebati
Genre: Sports
Rating: 4.4
Price: 449.0

-----------------------------
Title: LOSE FAT, GET FITTR: THE SIMPLE SCIENCE OF STAYING HEALTHY FOR LIFE
Author: Jitendra Chouksey
Genre: Sports
Rating: 4.6
Price: 223.25

-----------------------------
Title: The Haywire Heart: How too much exercise can kill you, and what you can do to protect your heart
A